In [1]:

import numpy as np

from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.linear_model import LogisticRegression
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from uni2ts.model.moirai import MoiraiModule
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
#import seaborn as sns

from encoder import MoiraiEncoder

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils

In [2]:
PATCH_SIZES = [8, 16, 32, 64]

SIZE = "small"
DEVICE = "cuda"
NUM_VARS = 6

DO_MASK = True
KEEP_MASK_EMBEDDING = True

In [3]:
def preprocess_data(
    data: np.ndarray,
    *,
    device: str | torch.device = "cpu",
    dtype: torch.dtype = torch.float32,
):
    """
    data: np.ndarray of shape (N, T, V) = (n_individual, time, variate)
    Assumes NO missing values and NO padding in the raw data.
    """

    if not isinstance(data, np.ndarray):
        raise TypeError("data must be a numpy.ndarray")
    if data.ndim != 3:
        raise ValueError(f"Expected shape (N,T,V), got {data.shape}")

    N, T, V = data.shape

    # (N,T,V)
    past_target = torch.as_tensor(data, dtype=dtype, device=device)

    # observed mask: (N,T,V) all True no missing values
    past_observed_target = torch.ones((N, T, V), dtype=torch.bool, device=device)

    # padding mask: (N,T) all if no padding
    past_is_pad = torch.zeros((N, T), dtype=torch.bool, device=device)

    return past_target, past_observed_target, past_is_pad

In [4]:
def apply_pooling_pt(Z_tensor, method, num_vars=NUM_VARS):
    N, S, F = Z_tensor.shape
    P = S // num_vars # Calcul automatique du nombre de patches par variable
    
    # On reshape le tenseur pour séparer les Variables et les Patches
    # Forme résultante : (Batch, Variables, Patches, Features)
    Z_reshaped = Z_tensor.view(N, num_vars, P, F)
    
    # Basique et Global
    if method == "flatten":
        return Z_tensor.reshape(N, -1)
        
    elif method == "global_mean":
        return Z_tensor.mean(dim=1)
        
    elif method == "global_max":
        return Z_tensor.max(dim=1).values
        
    elif method == "global_min":
        return Z_tensor.min(dim=1).values
    
    elif method == "global_mean_max_min":
        return torch.cat([
            Z_tensor.mean(dim=1),
            Z_tensor.max(dim=1).values,
            Z_tensor.min(dim=1).values
        ], dim=1)

    # Pooling sur les Patches (on garde les variables distinctes) ---
    # Réduction sur la dimension 2 (Patches). Résultat : (N, num_vars, F), puis on aplatit
    elif method == "mean_over_patches":
        return Z_reshaped.mean(dim=2).reshape(N, -1)
        
    elif method == "max_over_patches":
        return Z_reshaped.max(dim=2).values.reshape(N, -1)
        
    elif method == "min_over_patches":
        return Z_reshaped.min(dim=2).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_patches":
        p_mean = Z_reshaped.mean(dim=2).reshape(N, -1)
        p_max  = Z_reshaped.max(dim=2).values.reshape(N, -1)
        p_min  = Z_reshaped.min(dim=2).values.reshape(N, -1)
        return torch.cat([p_mean, p_max, p_min], dim=1)

    # Pooling sur les Variables (on synchronise les patches entre variables) ---
    # Réduction sur la dimension 1 (Variables). Résultat : (N, P, F), puis on aplatit
    elif method == "mean_over_variables":
        return Z_reshaped.mean(dim=1).reshape(N, -1)
        
    elif method == "max_over_variables":
        return Z_reshaped.max(dim=1).values.reshape(N, -1)
        
    elif method == "min_over_variables":
        return Z_reshaped.min(dim=1).values.reshape(N, -1)
        
    elif method == "mean_max_min_over_variables":
        v_mean = Z_reshaped.mean(dim=1).reshape(N, -1)
        v_max  = Z_reshaped.max(dim=1).values.reshape(N, -1)
        v_min  = Z_reshaped.min(dim=1).values.reshape(N, -1)
        return torch.cat([v_mean, v_max, v_min], dim=1)

    else:
        raise ValueError(f"Method {method} unknow")

pooling_methods = [
    "flatten", 
    "global_mean", "global_max", "global_min", "global_mean_max_min",
    "mean_over_patches", "max_over_patches", "min_over_patches", "mean_max_min_over_patches",
    "mean_over_variables", "max_over_variables", "min_over_variables", "mean_max_min_over_variables"
]

In [5]:
alphas_to_test = [0.1,1,10,30,100,300,1000,10000]

models = {
    'Ridge': RidgeClassifierCV(alphas=alphas_to_test, cv=3),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

# Data

Import

In [6]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

NUM_CLASSES = len(torch.unique(y_train_tensor))

Prepare data

In [7]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=DEVICE)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=DEVICE)
)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

num_classes = len(torch.unique(y_train_tensor))
NUM_VARS = 6

print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

torch.Size([2459, 36, 6])
torch.Size([2459, 36, 6])
torch.Size([2459, 36])


Creat Z_train and Z_test for each patch size

In [8]:
Z_train_patch = {}
Z_test_patch = {}
for patch in PATCH_SIZES:
    
    # 1. Définir les paramètres et l'encodeur pour la taille de patch actuelle
    MODEL_PARAMS = {
        "patch_size": patch,
        "num_samples": 100,
        "target_dim": 6,
        "feat_dynamic_real_dim": 0,
        "past_feat_dynamic_real_dim": 0,
    }
    if DO_MASK == True:
        prediction_length = patch
    else:
        prediction_length = 0
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=prediction_length,
        context_length=36,
        **MODEL_PARAMS,
    )
    encoder.eval()
    encoder.to(DEVICE)
    
    # 2. Extraire les embeddings (Z_train et Z_test)
    with torch.no_grad():
        Z_train = encoder(
            past_target=X_train_target_tensor,
            past_observed_target=X_train_is_target_observed,
            past_is_pad=X_train_is_target_padded,
        )
        Z_test = encoder(
            past_target=X_test_target_tensor,
            past_observed_target=X_test_is_target_observed,
            past_is_pad=X_test_is_target_padded,
        )
        
    if DO_MASK and not KEEP_MASK_EMBEDDING:
        N, S, F = Z_train.shape
        P = S // NUM_VARS  # Nombre de patchs par variable (Contexte + 1 Masque)
        
        # On reshape pour séparer les variables et les patchs : (Batch, 6 vars, Patchs, Features)
        Z_train_reshaped = Z_train.view(N, NUM_VARS, P, F)
        Z_test_reshaped = Z_test.view(Z_test.shape[0], NUM_VARS, P, F) # Attention à bien récupérer N_test = Z_test.shape[0]
        
        # On slice pour enlever le dernier patch (:-1) sur la dimension des patchs (dim=2)
        Z_train_no_mask = Z_train_reshaped[:, :, :-1, :]
        Z_test_no_mask = Z_test_reshaped[:, :, :-1, :]
        
        # On remet sous la forme attendue (Batch, Séquence_sans_masques, Features)
        Z_train = Z_train_no_mask.reshape(N, -1, F)
        Z_test = Z_test_no_mask.reshape(Z_test.shape[0], -1, F)



    Z_train_patch[patch] = Z_train
    Z_test_patch[patch] = Z_test

In [9]:

print(f"{'Patch Size':<12} | {'Train Shape':<25} | {'Test Shape':<25}")
for patch in PATCH_SIZES:
    train_tensor = Z_train_patch[patch]
    test_tensor = Z_test_patch[patch]
    print(f"{patch:<12} | {str(list(train_tensor.shape)):<25} | {str(list(test_tensor.shape)):<25}")

Patch Size   | Train Shape               | Test Shape               
8            | [2459, 36, 384]           | [2466, 36, 384]          
16           | [2459, 24, 384]           | [2466, 24, 384]          
32           | [2459, 18, 384]           | [2466, 18, 384]          
64           | [2459, 12, 384]           | [2466, 12, 384]          


# Basic pooling with logistic regression and random forest

## Single patch size pooling

We started by applying simple pooling techniques (max, min, or mean) and comparing their performance across various patch sizes. Specifically, we evaluated whether to apply this pooling globally across all patches, locally within a single series, or across corresponding patches from all series.

### Training

We creat a dataframe to stock the results and we fill it

In [10]:
results_dfs = {
    model_name: pd.DataFrame(index=pooling_methods, columns=PATCH_SIZES)
    for model_name in models.keys()
}
for df in results_dfs.values():
    df.index.name = "Pooling \\ Patch"

We train

In [11]:
for patch in PATCH_SIZES:
    Z_train = Z_train_patch[patch].to(DEVICE)
    Z_test = Z_test_patch[patch].to(DEVICE)
    
    for pool in pooling_methods:
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_pool)
        X_test_scaled = scaler.transform(X_test_pool)
        
        for model_name, clf in models.items():
            clf.fit(X_train_scaled, y_train)
            
            y_pred = clf.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)
            
            results_dfs[model_name].loc[pool, patch] = round(acc, 3)


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.96243e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.99794e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.95261e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditio

### Results

In [12]:
for model_name, df in results_dfs.items():
    print(f"RÉSULTATS POUR : {model_name.upper()}")
    display(df.astype(float))
    
    best_acc = df.max().max()
    best_pool, best_patch = df.astype(float).stack().idxmax()
    print(f"Meilleure combinaison = Patch: {best_patch} | Pooling: '{best_pool}' | Acc: {best_acc:.3f}\n")

RÉSULTATS POUR : RIDGE


,8,16,32,64
Pooling \ Patch,,,,
flatten,0.608,0.625,0.620,0.633
global_mean,0.586,0.590,0.583,0.598
global_max,0.544,0.552,0.536,0.578
global_min,0.545,0.553,0.538,0.565
global_mean_max_min,0.577,0.581,0.577,0.594
mean_over_patches,0.617,0.623,0.619,0.636
max_over_patches,0.590,0.610,0.601,0.632
min_over_patches,0.589,0.605,0.603,0.632
mean_max_min_over_patches,0.611,0.631,0.626,0.633


Meilleure combinaison = Patch: 64 | Pooling: 'mean_over_patches' | Acc: 0.636

RÉSULTATS POUR : RANDOMFOREST


,8,16,32,64
Pooling \ Patch,,,,
flatten,0.567,0.553,0.551,0.592
global_mean,0.574,0.576,0.559,0.579
global_max,0.546,0.522,0.521,0.564
global_min,0.523,0.528,0.515,0.565
global_mean_max_min,0.571,0.573,0.547,0.580
mean_over_patches,0.591,0.578,0.559,0.602
max_over_patches,0.565,0.566,0.552,0.597
min_over_patches,0.559,0.559,0.552,0.597
mean_max_min_over_patches,0.581,0.573,0.564,0.599


Meilleure combinaison = Patch: 64 | Pooling: 'mean_over_patches' | Acc: 0.602



df_ridge_plot = df_results_ridge.astype(float)
df_rf_plot = df_results_rf.astype(float)


fig, axes = plt.subplots(1, 2, figsize=(20, 10), sharey=True)

cmap = "YlGnBu"

sns.heatmap(df_ridge_plot, 
            annot=True,
            fmt=".3f",
            cmap=cmap,
            linewidths=0.5,
            ax=axes[0],
            cbar_kws={'label': 'Accuracy'})
axes[0].set_title("Single Patch Pooling - Accuracy (Ridge)", fontsize=16, fontweight='bold', pad=15)
axes[0].set_xlabel("Patch Size", fontsize=14)
axes[0].set_ylabel("Pooling Method", fontsize=14)
axes[0].tick_params(axis='both', which='major', labelsize=12)

sns.heatmap(df_rf_plot, 
            annot=True, 
            fmt=".3f", 
            cmap=cmap, 
            linewidths=0.5, 
            ax=axes[1], 
            cbar_kws={'label': 'Accuracy'})
axes[1].set_title("Single Patch Pooling - Accuracy (Random Forest)", fontsize=16, fontweight='bold', pad=15)
axes[1].set_xlabel("Patch Size", fontsize=14)
axes[1].tick_params(axis='both', which='major', labelsize=12)

plt.tight_layout()

plt.show()

Pooling over patches rather than series produces the highest accuracy.

## Multi Patch pooling

### Training

In [13]:
df_multiscale_results = pd.DataFrame(index=pooling_methods, columns=models.keys())
df_multiscale_results.index.name = "Pooling Method"

for pool in pooling_methods:
    X_train_multi = []
    X_test_multi = []
    
    for patch in PATCH_SIZES:
        Z_train = Z_train_patch[patch].to(DEVICE)
        Z_test = Z_test_patch[patch].to(DEVICE)
        
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()
        
        X_train_multi.append(X_train_pool)
        X_test_multi.append(X_test_pool)
        
    X_train_combined = np.concatenate(X_train_multi, axis=1)
    X_test_combined = np.concatenate(X_test_multi, axis=1)
    
    scaler = StandardScaler()
    X_train_combined_scaled = scaler.fit_transform(X_train_combined)
    X_test_combined_scaled = scaler.transform(X_test_combined)

    for model_name, clf in models.items():
        clf.fit(X_train_combined_scaled, y_train)
        
        y_pred = clf.predict(X_test_combined_scaled)
        acc = accuracy_score(y_test, y_pred)
        
        df_multiscale_results.loc[pool, model_name] = round(acc, 3)

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.40241e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.42041e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.37498e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: Ill-conditio

### Results

In [14]:
display(df_multiscale_results.astype(float))

best_acc = df_multiscale_results.max().max()
best_pool = df_multiscale_results.max(axis=1).idxmax()
best_model = df_multiscale_results.max(axis=0).idxmax()

print("MEILLEURE COMBINAISON MULTI-ÉCHELLE :")
print(f"Modèle : '{best_model}' | Pooling : '{best_pool}'")
print(f"Précision maximale : {best_acc:.3f}")

,Ridge,RandomForest
Pooling Method,,
flatten,0.649,0.580
global_mean,0.623,0.594
global_max,0.591,0.564
global_min,0.587,0.562
global_mean_max_min,0.612,0.590
mean_over_patches,0.658,0.608
max_over_patches,0.659,0.591
min_over_patches,0.650,0.596
mean_max_min_over_patches,0.664,0.599


MEILLEURE COMBINAISON MULTI-ÉCHELLE :
Modèle : 'Ridge' | Pooling : 'mean_max_min_over_patches'
Précision maximale : 0.664


## Conclusion

pool over patch -> good
16 -> (max, mean, min) > flatten -> position not importante
multi patch -> good

# Advanced Pooling

## Attention Pooling

In [58]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm

def train_and_evaluate_attention(
    model, 
    train_loader, 
    test_loader, 
    epochs=1000, 
    lr=1e-4, 
    weight_decay=1e-4,
    optimizer_name="Adam",
    scheduler_patience=50,
    early_stopping_patience=300,
    device="cuda",
    verbose=True
):
    """
    Boucle d'entraînement robuste spécifiquement adaptée pour les modèles d'attention
    qui prennent des embeddings pré-calculés (Z_concat) en entrée.
    """
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    # 1. Sélection de l'optimiseur
    if optimizer_name.lower() == "adamw":
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Optimizer {optimizer_name} non supporté.")
        
    # 2. Scheduler
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=scheduler_patience)
    
    # 3. Suivi et Checkpointing
    history = {'train_loss': [], 'test_loss': [], 'test_acc': [], 'lr': []}
    best_acc = 0.0
    best_model_weights = copy.deepcopy(model.state_dict())
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0.0
        
        # --- ENTRAÎNEMENT ---
        # Remarquez que le DataLoader renvoie (batch_z, batch_y)
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{epochs} [Train]", leave=False, disable=not verbose)
        for batch_z, batch_y in train_bar:
            batch_z, batch_y = batch_z.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            logits = model(batch_z)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item() * batch_y.size(0)
            
        avg_train_loss = total_train_loss / len(train_loader.dataset)
        
        # --- ÉVALUATION ---
        model.eval()
        total_test_loss = 0.0
        correct, total = 0, 0
        
        test_bar = tqdm(test_loader, desc=f"Epoch {epoch+1:03d}/{epochs} [Test]", leave=False, disable=not verbose)
        with torch.no_grad():
            for batch_z, batch_y in test_bar:
                batch_z, batch_y = batch_z.to(device), batch_y.to(device)
                
                logits = model(batch_z)
                loss = criterion(logits, batch_y)
                total_test_loss += loss.item() * batch_y.size(0)
                
                predictions = torch.argmax(logits, dim=-1)
                correct += (predictions == batch_y).sum().item()
                total += batch_y.size(0)
                
        avg_test_loss = total_test_loss / total
        test_acc = correct / total
        
        # Mise à jour du Scheduler
        scheduler.step(test_acc)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Sauvegarde de l'historique
        history['train_loss'].append(avg_train_loss)
        history['test_loss'].append(avg_test_loss)
        history['test_acc'].append(test_acc)
        history['lr'].append(current_lr)
        
        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"Epoch {epoch+1:03d}/{epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} | "
                  f"Test Loss: {avg_test_loss:.4f} | "
                  f"Test Acc: {test_acc:.4f} | "
                  f"LR: {current_lr:.2e}")
            
        # --- EARLY STOPPING & CHECKPOINTING ---
        if test_acc > best_acc:
            best_acc = test_acc
            best_model_weights = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= early_stopping_patience:
            if verbose:
                print(f"\n[!] Early stopping déclenché après {epoch+1} époques.")
            break
            
    # Chargement des poids du MEILLEUR modèle avant de le retourner
    model.load_state_dict(best_model_weights)
    if verbose:
        print(f"--- Entraînement Terminé. Meilleure Test Acc: {best_acc:.4f} ---")
        
    return history, model

In [25]:
class DynamicAttentionClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, patches_counts, in_features=384, mode="global"):
        """
        patches_counts : liste du nombre de patchs par échelle (ex: [30, 18, 12, 6])
        mode : "global" (1), "per_scale" (4), "per_variable" (6), ou "per_both" (24)
        """
        super().__init__()
        self.num_vars = num_vars
        self.patches_counts = patches_counts
        self.num_scales = len(patches_counts)
        self.mode = mode
        
        if mode == "global":
            self.context = nn.Parameter(torch.randn(in_features) * 0.1)
            
        elif mode == "per_scale":
            self.context = nn.Parameter(torch.randn(self.num_scales, in_features) * 0.1)
            
        elif mode == "per_variable":
            self.context = nn.Parameter(torch.randn(self.num_vars, in_features) * 0.1)
            
        elif mode == "per_both":
            self.context = nn.Parameter(torch.randn(self.num_scales, self.num_vars, in_features) * 0.1)
            
            
        self.dropout = nn.Dropout(0.1)
        self.linear = nn.LazyLinear(num_classes)

        
    def forward(self, x):
        B = x.shape[0]
        F = x.shape[-1]
        
        split_sizes = [p * self.num_vars for p in self.patches_counts]
        x_splits = torch.split(x, split_sizes, dim=1)
        
        reshaped_splits = []
        for i, p in enumerate(self.patches_counts):
            reshaped_splits.append(x_splits[i].view(B, self.num_vars, p, F))
            
        if self.mode == "global":
            scores = torch.matmul(x, self.context)
            weights = torch.softmax(scores, dim=1).unsqueeze(-1)
            pooled = (weights * x).sum(dim=1)
            
        elif self.mode == "per_scale":
            pooled_list = []
            for i, xi in enumerate(reshaped_splits):
                xi_flat = xi.view(B, -1, F)
                scores = torch.matmul(xi_flat, self.context[i])
                weights = torch.softmax(scores, dim=1).unsqueeze(-1)
                pooled_list.append((weights * xi_flat).sum(dim=1))
            
            pooled = torch.cat(pooled_list, dim=1)
            
        elif self.mode == "per_variable":
            x_vars = torch.cat(reshaped_splits, dim=2)
            
            scores = (x_vars * self.context.view(1, self.num_vars, 1, F)).sum(dim=-1)
            weights = torch.softmax(scores, dim=2).unsqueeze(-1)
            pooled_vars = (weights * x_vars).sum(dim=2)
            pooled = pooled_vars.view(B, -1)
            
        elif self.mode == "per_both":
            pooled_list = []
            for i, xi in enumerate(reshaped_splits):
                scores = (xi * self.context[i].view(1, self.num_vars, 1, F)).sum(dim=-1)
                weights = torch.softmax(scores, dim=2).unsqueeze(-1)
                pooled_vars = (weights * xi).sum(dim=2)
                pooled_list.append(pooled_vars)
                
            pooled_all = torch.stack(pooled_list, dim=1)
            pooled = pooled_all.view(B, -1)
        
            
        return self.linear(self.dropout(pooled))

In [26]:
patch_sizes = [8, 16, 32, 64]
NUM_VARS = 6

patches_counts = [Z_train_patch[p].shape[1] // NUM_VARS for p in patch_sizes]
print(f"Nomber of patch for each scale : {patches_counts}")

Z_train_concat = torch.cat([Z_train_patch[p].to(DEVICE) for p in patch_sizes], dim=1)
Z_test_concat  = torch.cat([Z_test_patch[p].to(DEVICE) for p in patch_sizes], dim=1)

print(f"Shape Z_train: {Z_train_concat.shape}")

Nomber of patch for each scale : [6, 4, 3, 2]
Shape Z_train: torch.Size([2459, 90, 384])


In [27]:
BATCH_SIZE = 64*2**2

attention_modes = ['global', 'per_scale', 'per_variable', 'per_both']
df_results = pd.DataFrame(index=attention_modes, columns=["Test Accuracy"])
df_results.index.name = "Attention mode"

train_dataset = TensorDataset(Z_train_concat, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(Z_test_concat, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [28]:
mode = "global"
model_dyn = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(DEVICE)

# For LazyLinear
_ = model_dyn(Z_train_concat[:2].to(DEVICE))


history_dyn, best_model_dyn = train_and_evaluate_attention(
    model=model_dyn,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=1000,
    lr=1e-4,
    weight_decay=1e-6,
    scheduler_patience=15,
    early_stopping_patience=40,
    device=DEVICE,
    verbose=True
)

best_acc_dyn = max(history_dyn['test_acc'])
df_results.loc[mode, "Test Accuracy"] = round(best_acc_dyn, 3)

Epoch 001/1000 | Train Loss: 2.6142 | Test Loss: 2.5905 | Test Acc: 0.0369 | LR: 1.00e-04


Epoch 010/1000 | Train Loss: 2.3196 | Test Loss: 2.2998 | Test Acc: 0.3248 | LR: 1.00e-04


Epoch 020/1000 | Train Loss: 2.1135 | Test Loss: 2.1015 | Test Acc: 0.3159 | LR: 1.00e-04


Epoch 030/1000 | Train Loss: 2.0389 | Test Loss: 2.0324 | Test Acc: 0.3155 | LR: 5.00e-05


Epoch 040/1000 | Train Loss: 2.0093 | Test Loss: 2.0003 | Test Acc: 0.3155 | LR: 5.00e-05



[!] Early stopping déclenché après 49 époques.
--- Entraînement Terminé. Meilleure Test Acc: 0.3268 ---


In [29]:
mode = "per_scale"
model_dyn = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(DEVICE)

# For LazyLinear
_ = model_dyn(Z_train_concat[:2].to(DEVICE))


history_dyn, best_model_dyn = train_and_evaluate_attention(
    model=model_dyn,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=1000,
    lr=1e-4,
    weight_decay=1e-6,
    scheduler_patience=15,
    early_stopping_patience=40,
    device=DEVICE,
    verbose=True
)

best_acc_dyn = max(history_dyn['test_acc'])
df_results.loc[mode, "Test Accuracy"] = round(best_acc_dyn, 3)

Epoch 001/1000 | Train Loss: 2.4681 | Test Loss: 2.3717 | Test Acc: 0.2109 | LR: 1.00e-04


Epoch 010/1000 | Train Loss: 1.8847 | Test Loss: 1.8706 | Test Acc: 0.4051 | LR: 1.00e-04


Epoch 020/1000 | Train Loss: 1.7168 | Test Loss: 1.7110 | Test Acc: 0.4599 | LR: 1.00e-04


Epoch 030/1000 | Train Loss: 1.6071 | Test Loss: 1.6054 | Test Acc: 0.4915 | LR: 1.00e-04


Epoch 040/1000 | Train Loss: 1.5244 | Test Loss: 1.5295 | Test Acc: 0.5178 | LR: 1.00e-04


...

Epoch 200/1000 | Train Loss: 1.1049 | Test Loss: 1.2302 | Test Acc: 0.5961 | LR: 1.00e-04


Epoch 210/1000 | Train Loss: 1.0906 | Test Loss: 1.2262 | Test Acc: 0.5961 | LR: 5.00e-05


Epoch 220/1000 | Train Loss: 1.0854 | Test Loss: 1.2245 | Test Acc: 0.5973 | LR: 5.00e-05


Epoch 230/1000 | Train Loss: 1.0806 | Test Loss: 1.2231 | Test Acc: 0.5961 | LR: 2.50e-05



[!] Early stopping déclenché après 233 époques.
--- Entraînement Terminé. Meilleure Test Acc: 0.5973 ---


In [30]:
mode = "per_variable"
model_dyn = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(DEVICE)

# For LazyLinear
_ = model_dyn(Z_train_concat[:2].to(DEVICE))


history_dyn, best_model_dyn = train_and_evaluate_attention(
    model=model_dyn,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=1000,
    lr=1e-4,
    weight_decay=1e-6,
    scheduler_patience=15,
    early_stopping_patience=40,
    device=DEVICE,
    verbose=True
)

best_acc_dyn = max(history_dyn['test_acc'])
df_results.loc[mode, "Test Accuracy"] = round(best_acc_dyn, 3)

Epoch 001/1000 | Train Loss: 2.5083 | Test Loss: 2.3824 | Test Acc: 0.3037 | LR: 1.00e-04


Epoch 010/1000 | Train Loss: 1.8837 | Test Loss: 1.8722 | Test Acc: 0.3832 | LR: 1.00e-04


Epoch 020/1000 | Train Loss: 1.6710 | Test Loss: 1.6750 | Test Acc: 0.4886 | LR: 1.00e-04


Epoch 030/1000 | Train Loss: 1.5320 | Test Loss: 1.5469 | Test Acc: 0.5219 | LR: 1.00e-04


Epoch 040/1000 | Train Loss: 1.4301 | Test Loss: 1.4615 | Test Acc: 0.5357 | LR: 1.00e-04


...

Epoch 160/1000 | Train Loss: 0.9811 | Test Loss: 1.1846 | Test Acc: 0.6119 | LR: 1.00e-04


Epoch 170/1000 | Train Loss: 0.9571 | Test Loss: 1.1782 | Test Acc: 0.6115 | LR: 1.00e-04


Epoch 180/1000 | Train Loss: 0.9435 | Test Loss: 1.1745 | Test Acc: 0.6107 | LR: 5.00e-05


Epoch 190/1000 | Train Loss: 0.9362 | Test Loss: 1.1721 | Test Acc: 0.6115 | LR: 2.50e-05



[!] Early stopping déclenché après 197 époques.
--- Entraînement Terminé. Meilleure Test Acc: 0.6152 ---


In [31]:
mode = "per_both"
model_dyn = DynamicAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patches_counts=patches_counts, 
    mode=mode
).to(DEVICE)

# For LazyLinear
_ = model_dyn(Z_train_concat[:2].to(DEVICE))


history_dyn, best_model_dyn = train_and_evaluate_attention(
    model=model_dyn,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=1000,
    lr=1e-4,
    weight_decay=1e-6,
    scheduler_patience=15,
    early_stopping_patience=40,
    device=DEVICE,
    verbose=True
)

best_acc_dyn = max(history_dyn['test_acc'])
df_results.loc[mode, "Test Accuracy"] = round(best_acc_dyn, 3)

Epoch 001/1000 | Train Loss: 2.4188 | Test Loss: 2.0671 | Test Acc: 0.3244 | LR: 1.00e-04


Epoch 010/1000 | Train Loss: 1.3550 | Test Loss: 1.3898 | Test Acc: 0.5612 | LR: 1.00e-04


Epoch 020/1000 | Train Loss: 1.1291 | Test Loss: 1.2353 | Test Acc: 0.6139 | LR: 1.00e-04


Epoch 030/1000 | Train Loss: 1.0011 | Test Loss: 1.1667 | Test Acc: 0.6354 | LR: 1.00e-04


Epoch 040/1000 | Train Loss: 0.9049 | Test Loss: 1.1274 | Test Acc: 0.6411 | LR: 1.00e-04


...

Epoch 060/1000 | Train Loss: 0.7637 | Test Loss: 1.0869 | Test Acc: 0.6423 | LR: 1.00e-04


Epoch 070/1000 | Train Loss: 0.7098 | Test Loss: 1.0765 | Test Acc: 0.6436 | LR: 5.00e-05


Epoch 080/1000 | Train Loss: 0.6844 | Test Loss: 1.0724 | Test Acc: 0.6440 | LR: 5.00e-05


Epoch 090/1000 | Train Loss: 0.6653 | Test Loss: 1.0704 | Test Acc: 0.6464 | LR: 2.50e-05



[!] Early stopping déclenché après 91 époques.
--- Entraînement Terminé. Meilleure Test Acc: 0.6468 ---


In [32]:
display(df_results)

,Test Accuracy
Attention mode,
global,0.327
per_scale,0.597
per_variable,0.615
per_both,0.647


In [ ]:
import torch
import torch.nn as nn

class SingleScaleAttentionClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, in_features=384, mode="shared_context"):
        """
        mode: 
          - "shared_context": 1 seul vecteur contexte pour les 6 variables.
          - "independent_context": 6 vecteurs contextes distincts (1 par variable).
        """
        super().__init__()
        self.num_vars = num_vars
        self.mode = mode

        if mode == "shared_context":
            # Un seul vecteur de taille F (384)
            self.context = nn.Parameter(torch.randn(in_features) * 0.1)
            
        elif mode == "independent_context":
            # Une matrice de taille (V, F) -> 6 vecteurs de taille 384
            self.context = nn.Parameter(torch.randn(num_vars, in_features) * 0.1)
            

        self.dropout = nn.Dropout(0.3)
        
        linear_in = num_vars * in_features
        self.linear = nn.Linear(linear_in, num_classes)
        

    def forward(self, x):
        # x est le tenseur d'une seule échelle (ex: patch_size=16)
        # x shape: (Batch, Séquence Totale, Features)
        B = x.shape[0]
        S = x.shape[1]
        F = x.shape[-1]
        
        P = S // self.num_vars

        x_reshaped = x.view(B, self.num_vars, P, F)
        
        # 2. Calcul des Scores d'Attention
        if self.mode == "shared_context":
            # Produit scalaire de tous les patchs avec le MÊME vecteur contexte
            # x_reshaped: (B, V, P, F) | context: (F)
            scores = torch.matmul(x_reshaped, self.context) # Résultat: (B, V, P)
            
        elif self.mode == "independent_context":

            context_view = self.context.view(1, self.num_vars, 1, F)
            scores = (x_reshaped * context_view).sum(dim=-1) # Résultat: (B, V, P)
            
        weights = torch.softmax(scores, dim=2).unsqueeze(-1) # shape: (B, V, P, 1)
        
        pooled_vars = (weights * x_reshaped).sum(dim=2) # shape: (B, V, F)
        
        pooled = pooled_vars.view(B, -1) # shape: (B, V * F)
        
        return self.linear(self.dropout(pooled))

In [ ]:
PATCH_CHOISI = 8

Z_train_16 = Z_train_patch[PATCH_CHOISI].to(DEVICE)
Z_test_16 = Z_test_patch[PATCH_CHOISI].to(DEVICE)

BATCH_SIZE = 256
train_dataset = TensorDataset(Z_train_16, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(Z_test_16, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_shared = SingleScaleAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    mode="shared_context"
).to(DEVICE)

history_shared, _ = train_and_evaluate_attention(
    model=model_shared, train_loader=train_loader, test_loader=test_loader,
    epochs=2000, lr=1e-4, weight_decay=0.0005, early_stopping_patience=300, device=DEVICE, scheduler_patience=20
)


model_indep = SingleScaleAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    mode="independent_context"
).to(DEVICE)

history_indep, _ = train_and_evaluate_attention(
    model=model_indep, train_loader=train_loader, test_loader=test_loader,
    epochs=2000, lr=1e-4, weight_decay=0.0005, early_stopping_patience=300, device=DEVICE, scheduler_patience=20
)

✅ Modèle SingleScaleAttention [SHARED_CONTEXT] créé !
   ➔ Couche Linéaire : Entrée (2304) -> Sortie (14)
   ➔ Paramètres de la tête : 32,270

🚀 Entraînement : Shared Context


Epoch 001/2000 | Train Loss: 2.6483 | Test Loss: 2.4879 | Test Acc: 0.2847 | LR: 1.00e-04


Epoch 010/2000 | Train Loss: 1.9010 | Test Loss: 1.8901 | Test Acc: 0.3929 | LR: 1.00e-04


Epoch 020/2000 | Train Loss: 1.7216 | Test Loss: 1.7244 | Test Acc: 0.4578 | LR: 1.00e-04


Epoch 030/2000 | Train Loss: 1.6059 | Test Loss: 1.6161 | Test Acc: 0.4931 | LR: 1.00e-04


...

Epoch 680/2000 | Train Loss: 0.9581 | Test Loss: 1.2005 | Test Acc: 0.6046 | LR: 7.98e-08


Epoch 690/2000 | Train Loss: 0.9581 | Test Loss: 1.2005 | Test Acc: 0.6046 | LR: 5.59e-08


Epoch 700/2000 | Train Loss: 0.9573 | Test Loss: 1.2005 | Test Acc: 0.6046 | LR: 5.59e-08


Epoch 710/2000 | Train Loss: 0.9699 | Test Loss: 1.2005 | Test Acc: 0.6046 | LR: 5.59e-08



[!] Early stopping déclenché après 717 époques.
--- Entraînement Terminé. Meilleure Test Acc: 0.6046 ---


In [ ]:
import torch
import torch.nn as nn

class SingleScaleMultiHeadClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, in_features=384, num_heads=4, mode="shared_context"):
        """
        mode: 
          - "shared_context": 1 seul token [CLS] (Query) partagé pour lire les 6 variables.
          - "independent_context": 6 tokens [CLS] distincts (1 spécialiste par variable).
        """
        super().__init__()
        self.num_vars = num_vars
        self.mode = mode
        
        if mode == "shared_context":
            self.cls_token = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)
            
        elif mode == "independent_context":
            self.cls_token = nn.Parameter(torch.randn(num_vars, 1, in_features) * 0.02)

            
        self.mha = nn.MultiheadAttention(
            embed_dim=in_features,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True 
        )
            
        self.dropout = nn.Dropout(0.3)

        linear_in = num_vars * in_features
        self.linear = nn.Linear(linear_in, num_classes)


    def forward(self, x):
        B = x.shape[0]
        S = x.shape[1]
        F = x.shape[-1]

        P = S // self.num_vars

        x_reshaped = x.view(B, self.num_vars, P, F)
        

        kv = x_reshaped.view(B * self.num_vars, P, F)
        
        if self.mode == "shared_context":
            q = self.cls_token.expand(B * self.num_vars, -1, -1)
            
        elif self.mode == "independent_context":
            q_expanded = self.cls_token.unsqueeze(0).expand(B, -1, -1, -1)
            q = q_expanded.reshape(B * self.num_vars, 1, F)
            

        attn_output, attn_weights = self.mha(query=q, key=kv, value=kv)
        

        pooled_flat = attn_output.squeeze(1) # shape: (B * V, F)
        
        pooled_vars = pooled_flat.view(B, self.num_vars, F) # shape: (B, V, F)
        
        pooled = pooled_vars.view(B, -1) # shape: (B, V * F)
        

        return self.linear(self.dropout(pooled))

In [ ]:
PATCH_CHOISI = 16

Z_train_16 = Z_train_patch[PATCH_CHOISI].to(DEVICE)
Z_test_16 = Z_test_patch[PATCH_CHOISI].to(DEVICE)

BATCH_SIZE = 256
train_dataset = TensorDataset(Z_train_16, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = TensorDataset(Z_test_16, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model_mha = SingleScaleMultiHeadClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    num_heads=384, # 4 têtes d'attention
    mode="shared_context"
).to(DEVICE)

history_mha, best_mha = train_and_evaluate_attention(
    model=model_mha, 
    train_loader=train_loader, 
    test_loader=test_loader,
    epochs=1000, 
    lr=1e-4,              # LR un peu plus grand pour l'attention
    weight_decay=0.000005,    # Forte régularisation (vs Ridge)
    early_stopping_patience=100,
    scheduler_patience=30,
    device=DEVICE
)

✅ Modèle SingleScale Multi-Head Attention [SHARED_CONTEXT] créé !
   ➔ Couche Linéaire : Entrée (2304) -> Sortie (14)
   ➔ Paramètres de la tête : 32,270

🚀 Entraînement : Multi-Head Attention (Independent)


Epoch 001/1000 [Train]:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 001/1000 | Train Loss: 2.3189 | Test Loss: 2.0869 | Test Acc: 0.3175 | LR: 1.00e-04


Epoch 010/1000 | Train Loss: 1.5145 | Test Loss: 1.5172 | Test Acc: 0.5036 | LR: 1.00e-04


Epoch 020/1000 | Train Loss: 1.2453 | Test Loss: 1.2913 | Test Acc: 0.5807 | LR: 1.00e-04


...

Epoch 130/1000 | Train Loss: 0.1611 | Test Loss: 1.3668 | Test Acc: 0.6152 | LR: 4.90e-05


Epoch 140/1000 | Train Loss: 0.1321 | Test Loss: 1.4038 | Test Acc: 0.6127 | LR: 4.90e-05


Epoch 150/1000 | Train Loss: 0.1003 | Test Loss: 1.4435 | Test Acc: 0.6115 | LR: 4.90e-05


Epoch 160/1000 | Train Loss: 0.0884 | Test Loss: 1.4813 | Test Acc: 0.6127 | LR: 3.43e-05



[!] Early stopping déclenché après 162 époques.
--- Entraînement Terminé. Meilleure Test Acc: 0.6358 ---


In [ ]:
import torch
import torch.nn as nn

class HierarchicalAttentionClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, in_features=384, patch_mode="independent_context"):
        super().__init__()
        self.num_vars = num_vars
        self.patch_mode = patch_mode

        if patch_mode == "shared_context":
            self.patch_context = nn.Parameter(torch.randn(in_features) * 0.1)
        elif patch_mode == "independent_context":
            self.patch_context = nn.Parameter(torch.randn(num_vars, in_features) * 0.1)

            

        self.var_context = nn.Parameter(torch.randn(in_features) * 0.1)
        
        self.dropout = nn.Dropout(0.3)
        

        self.linear = nn.Linear(in_features, num_classes)
        


    def forward(self, x):
        B = x.shape[0]
        S = x.shape[1]
        F = x.shape[-1]
        
        P = S // self.num_vars
        x_reshaped = x.view(B, self.num_vars, P, F)
        
        if self.patch_mode == "shared_context":
            patch_scores = torch.matmul(x_reshaped, self.patch_context) # (B, V, P)
        elif self.patch_mode == "independent_context":
            context_view = self.patch_context.view(1, self.num_vars, 1, F)
            patch_scores = (x_reshaped * context_view).sum(dim=-1) # (B, V, P)
            
        patch_weights = torch.softmax(patch_scores, dim=2).unsqueeze(-1) # (B, V, P, 1)

        var_reprs = (patch_weights * x_reshaped).sum(dim=2) # shape: (B, V, F)
        

        var_scores = torch.matmul(var_reprs, self.var_context)
        

        var_weights = torch.softmax(var_scores, dim=1).unsqueeze(-1)

        global_repr = (var_weights * var_reprs).sum(dim=1) # shape: (B, F)

        return self.linear(self.dropout(global_repr))

In [ ]:
PATCH_CHOISI = 16

Z_train_16 = Z_train_patch[PATCH_CHOISI].to(DEVICE)
Z_test_16 = Z_test_patch[PATCH_CHOISI].to(DEVICE)

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(Z_train_16, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(Z_test_16, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

model_hierarchical = HierarchicalAttentionClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    patch_mode="independent_context" 
).to(DEVICE)

history_hierarchical, best_hierarchical = train_and_evaluate_attention(
    model=model_hierarchical, 
    train_loader=train_loader, 
    test_loader=test_loader,
    epochs=1000, 
    lr=1e-3,
    weight_decay=0.0000000000000005,
    early_stopping_patience=100,
    scheduler_patience=30,
    device=DEVICE
)

✅ Modèle Hierarchical Attention (Patch: INDEPENDENT_CONTEXT) créé !
   ➔ Couche Linéaire : Entrée (384) -> Sortie (14)
   ➔ Paramètres de la tête : 5,390

🚀 Entraînement : Hierarchical Attention (Patch -> Variables)


Epoch 001/1000 | Train Loss: 2.4749 | Test Loss: 2.2701 | Test Acc: 0.3508 | LR: 1.00e-03


Epoch 010/1000 | Train Loss: 1.7028 | Test Loss: 1.6611 | Test Acc: 0.4676 | LR: 1.00e-03


Epoch 020/1000 | Train Loss: 1.5129 | Test Loss: 1.4937 | Test Acc: 0.5211 | LR: 1.00e-03


Epoch 030/1000 | Train Loss: 1.4137 | Test Loss: 1.4149 | Test Acc: 0.5426 | LR: 1.00e-03


...

Epoch 130/1000 | Train Loss: 1.0941 | Test Loss: 1.3255 | Test Acc: 0.5681 | LR: 4.90e-04


Epoch 140/1000 | Train Loss: 1.0836 | Test Loss: 1.3272 | Test Acc: 0.5689 | LR: 4.90e-04


Epoch 150/1000 | Train Loss: 1.0757 | Test Loss: 1.3298 | Test Acc: 0.5714 | LR: 4.90e-04


Epoch 160/1000 | Train Loss: 1.0631 | Test Loss: 1.3311 | Test Acc: 0.5706 | LR: 3.43e-04



[!] Early stopping déclenché après 163 époques.
--- Entraînement Terminé. Meilleure Test Acc: 0.5775 ---


In [ ]:
class HierarchicalMultiHeadClassifier(nn.Module):
    def __init__(self, num_vars, num_classes, in_features=384, num_heads=4, patch_mode="independent_context"):
        super().__init__()
        self.num_vars = num_vars
        self.patch_mode = patch_mode
        

        if patch_mode == "shared_context":
            self.patch_cls = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)
        elif patch_mode == "independent_context":
            self.patch_cls = nn.Parameter(torch.randn(num_vars, 1, in_features) * 0.02)


        self.patch_mha = nn.MultiheadAttention(
            embed_dim=in_features, num_heads=num_heads, dropout=0.1, batch_first=True
        )


        self.var_cls = nn.Parameter(torch.randn(1, 1, in_features) * 0.02)

        self.var_mha = nn.MultiheadAttention(
            embed_dim=in_features, num_heads=num_heads, dropout=0.1, batch_first=True
        )

        self.dropout = nn.Dropout(0.1)

        self.linear = nn.Linear(in_features, num_classes)


    def forward(self, x):
        B = x.shape[0]
        S = x.shape[1]
        F = x.shape[-1]
        
        P = S // self.num_vars
        
        x_reshaped = x.view(B, self.num_vars, P, F)
        kv_patch = x_reshaped.view(B * self.num_vars, P, F) # (B*V, P, F)
        
        if self.patch_mode == "shared_context":
            q_patch = self.patch_cls.expand(B * self.num_vars, -1, -1)
        elif self.patch_mode == "independent_context":
            q_expanded = self.patch_cls.unsqueeze(0).expand(B, -1, -1, -1)
            q_patch = q_expanded.reshape(B * self.num_vars, 1, F)
            
        patch_attn_out, _ = self.patch_mha(query=q_patch, key=kv_patch, value=kv_patch)

        var_reprs = patch_attn_out.squeeze(1).view(B, self.num_vars, F) # (B, V, F)
        

        kv_var = var_reprs # (B, V, F)
        

        q_var = self.var_cls.expand(B, -1, -1) # (B, 1, F)

        var_attn_out, _ = self.var_mha(query=q_var, key=kv_var, value=kv_var)

        global_repr = var_attn_out.squeeze(1) # (B, F)

        return self.linear(self.dropout(global_repr))

In [ ]:
PATCH_CHOISI = 16

Z_train_16 = Z_train_patch[PATCH_CHOISI].to(DEVICE)
Z_test_16 = Z_test_patch[PATCH_CHOISI].to(DEVICE)

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(Z_train_16, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(Z_test_16, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

model_hierarchical_mha = HierarchicalMultiHeadClassifier(
    num_vars=NUM_VARS, 
    num_classes=num_classes, 
    num_heads=384,
    patch_mode="independent_context" 
).to(DEVICE)

history_h_mha, best_h_mha = train_and_evaluate_attention(
    model=model_hierarchical_mha, 
    train_loader=train_loader, 
    test_loader=test_loader,
    epochs=1000, 
    lr=1e-4,
    weight_decay=0.000005,
    early_stopping_patience=100,
    scheduler_patience=30,
    device=DEVICE
)

✅ Modèle Hierarchical Multi-Head (Patch Mode: INDEPENDENT_CONTEXT) créé !
   ➔ Couche Linéaire : Entrée (384) -> Sortie (14)
   ➔ Paramètres de la tête : 5,390

🚀 Entraînement : Hierarchical MULTI-HEAD Attention


Epoch 001/1000 [Train]:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 001/1000 | Train Loss: 2.4727 | Test Loss: 2.2479 | Test Acc: 0.3155 | LR: 1.00e-04


Epoch 010/1000 | Train Loss: 1.6166 | Test Loss: 1.6079 | Test Acc: 0.4643 | LR: 1.00e-04


Epoch 020/1000 | Train Loss: 1.4102 | Test Loss: 1.3955 | Test Acc: 0.5434 | LR: 1.00e-04


...

Epoch 130/1000 | Train Loss: 0.2825 | Test Loss: 1.7948 | Test Acc: 0.5624 | LR: 4.90e-05


Epoch 140/1000 | Train Loss: 0.2311 | Test Loss: 1.9178 | Test Acc: 0.5637 | LR: 4.90e-05


Epoch 150/1000 | Train Loss: 0.1790 | Test Loss: 2.0785 | Test Acc: 0.5523 | LR: 4.90e-05


Epoch 160/1000 | Train Loss: 0.1405 | Test Loss: 2.2487 | Test Acc: 0.5543 | LR: 4.90e-05



[!] Early stopping déclenché après 168 époques.
--- Entraînement Terminé. Meilleure Test Acc: 0.6095 ---


TODO :
- target_dim?
- result diff de maxime car moi moi pytorch pour reglog